In [1]:
import numpy as np
import pandas as pd
import random
import csv
import geopandas as gpd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, MaxAbsScaler
from sklearn.impute import KNNImputer
import sys
import pickle

sys.path.insert(0, "../../../Modules")

pd.set_option('display.max_columns', None)
random.seed(0)
np.random.seed(0)

from utils import *
from utils import process_greek

enc = 'utf-8'
shapefiles_folder = "C:/Users/dimit/Documents/noa hoard/Greece Shapefiles"

c:\Users\dimit\AppData\Local\Programs\Python\Python39\lib\site-packages\geopandas\_compat.py:112: UserWarning: The Shapely GEOS version (3.10.3-CAPI-1.16.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
NUTS0 = 'GR'
NUTS2 = 'Thessaly'

In [4]:
YEAR = 2023
MONTH = 'May'
PERIOD = '2nd'

In [5]:
model = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Linear_model.pkl', 'rb'))
scaler = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Scaler.pkl', 'rb'))
imputer = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Imputer.pkl', 'rb'))

In [6]:
data_test = read_data(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Dataset_{YEAR}-{MONTH}-{PERIOD}.csv')
data_test.head()

,x,y,dt_placement,nuts2,lau1,day,month,week,year,day_sin,day_cos,month_sin,month_cos,week_sin,week_cos,area,population,population_density,eq_distance,ndvi,ndmi,ndwi,ndbi,ndvi_mean,ndmi_mean,ndwi_mean,ndbi_mean,ndvi_std,ndmi_std,ndwi_std,ndbi_std,lst,lst_day,lst_night,lst_jan_day_mean,lst_jan_night_mean,lst_feb_day_mean,lst_feb_night_mean,lst_mar_day_mean,lst_mar_night_mean,lst_apr_day_mean,lst_apr_night_mean,acc_rainfall_1week,acc_rainfall_2week,acc_rainfall_jan,distance_to_coast,distance_to_river,slope_mean_1km,aspect_mean_200m,elevation_mean_1km,hillshade_mean_1km,fs_area_1km,flow_accu_200m,lc_prop1,lc_prop1_assessment,lc_prop2,lc_prop2_assessment,lc_prop3,lc_prop3_assessment,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,lw,qc,mosq_pred,mosq_previous,case
0,22.76105,39.69186,2023-05-16,θεσσαλιας,αγιας,16,5,20,2023,-0.101168,-0.994869,0.5,-0.866025,0.696551,-0.717507,661.8,10705.0,17.3,45.75488,0.599177,0.271008,-0.481473,-0.271008,0.589585,0.277525,-0.473957,-0.277525,0.045826,0.041758,0.032822,0.041758,15.958438,20.465625,11.451250,10.418186,4.056923,10.285211,3.423839,16.670680,6.227661,17.857476,7.916795,13.718427,27.686035,107.549129,8282.127022,1331.619869,1,123.585589,121.767859,183.036108,0.0,21.311709,31,90,30,90.0,30,90,10,10,1,6,6,2,0,16,0,0
1,22.72671,39.12560,2023-05-16,θεσσαλιας,αλμυρου,16,5,20,2023,-0.101168,-0.994869,0.5,-0.866025,0.696551,-0.717507,905.4,16004.0,20.6,45.24728,0.574347,0.252648,-0.472073,-0.252648,0.556614,0.239914,-0.459363,-0.239914,0.044810,0.051985,0.031783,0.051985,14.397917,18.883333,9.912500,11.219310,3.845804,10.731281,3.006374,16.701534,6.377584,19.115966,8.094541,9.583817,16.600280,219.051269,10017.207897,1610.666116,14,259.585198,342.571837,171.163634,0.0,3.415750,11,80,10,72.0,10,72,1,1,7,1,1,2,0,1,0,0
2,23.99842,39.24585,2023-05-16,θεσσαλιας,αλοννησου,16,5,20,2023,-0.101168,-0.994869,0.5,-0.866025,0.696551,-0.717507,129.6,3153.0,21.2,46.00175,-0.065445,0.278424,0.163445,-0.278424,-0.066080,0.279226,0.171660,-0.279226,0.003815,0.004745,0.008506,0.004745,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.026084,33.434825,249.360479,1484.683259,55302.082778,24,302.637680,165.397106,195.027810,0.0,1.676994,21,94,20,94.0,20,94,8,8,4,1,1,2,0,1,0,0
3,21.48510,39.29630,2023-05-16,θεσσαλιας,αργιθεας,16,5,20,2023,-0.101168,-0.994869,0.5,-0.866025,0.696551,-0.717507,372.9,3515.0,9.3,44.78626,0.456972,0.091266,-0.412161,-0.091266,0.465986,0.099252,-0.413601,-0.099252,0.074489,0.064996,0.046331,0.064996,13.103077,18.354615,7.851538,6.942786,1.831897,7.062325,-0.605892,13.120876,3.695211,13.925591,4.285066,8.407568,23.750692,183.897530,19327.278832,1925.318354,27,284.268894,1127.254453,234.678996,0.0,1.719393,21,88,20,88.0,20,88,8,8,4,1,1,2,0,32,0,0
4,22.93502,39.38117,2023-05-16,θεσσαλιας,βολου,16,5,20,2023,-0.101168,-0.994869,0.5,-0.866025,0.696551,-0.717507,385.6,138865.0,374.6,45.57293,0.490548,0.170045,-0.406931,-0.170045,0.480246,0.157946,-0.405062,-0.157946,0.036480,0.036939,0.026775,0.036939,16.065000,21.241667,10.888333,12.159420,4.198851,11.394699,3.845437,17.241419,5.994943,19.899661,8.124530,12.164361,27.286452,175.785715,5935.142744,3071.988689,5,143.181190,234.931813,174.426021,0.0,4.643743,31,91,30,91.0,30,91,10,10,1,6,6,2,0,38,0,0


In [7]:
X_test = data_test.select_dtypes(exclude=['object']).drop(columns = ['case'])
y_test = data_test['case']

X_test = scaler.transform(X_test)
X_test = imputer.fit_transform(X_test)

results_test = inference_lin_model(model, data_test, X_test, y_test)

In [8]:
results_test.drop(columns='case', inplace= True)
results_test

,x,y,lau1,day,month,year,score
0,22.34107,39.61048,λαρισαιων,16,5,2023,0.002149
1,22.54288,39.85090,τεμπων,16,5,2023,0.002085
2,22.29611,39.76297,τυρναβου,16,5,2023,0.001391
3,22.15952,39.95174,ελασσονας,16,5,2023,0.001116
4,21.89535,39.26951,καρδιτσας,16,5,2023,0.001022
5,21.77435,39.61293,τρικκαιων,16,5,2023,0.000917
6,21.48510,39.29630,αργιθεας,16,5,2023,0.000717
7,22.02629,39.60300,φαρκαδονας,16,5,2023,0.000464
8,22.51098,39.51390,κιλελερ,16,5,2023,0.000369
9,22.76105,39.69186,αγιας,16,5,2023,0.000267


In [9]:
bins_path = f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_Bins_{YEAR}.csv'

bins = []

with open(bins_path, mode='r', newline='') as file:
    reader = csv.reader(file)
    for row in reader:
        bins.extend(map(float, row))

print(bins)

[0.0, 0.0079983711252831, 0.0857458244919643, 0.5105258673268654, 0.8201202051121793, 0.9298314119184128, 1.0]


In [10]:
results_test['risk_class'] = pd.cut(results_test['score'], bins=bins, labels=False, include_lowest=True)
results_test

,x,y,lau1,day,month,year,score,risk_class
0,22.34107,39.61048,λαρισαιων,16,5,2023,0.002149,0
1,22.54288,39.85090,τεμπων,16,5,2023,0.002085,0
2,22.29611,39.76297,τυρναβου,16,5,2023,0.001391,0
3,22.15952,39.95174,ελασσονας,16,5,2023,0.001116,0
4,21.89535,39.26951,καρδιτσας,16,5,2023,0.001022,0
5,21.77435,39.61293,τρικκαιων,16,5,2023,0.000917,0
6,21.48510,39.29630,αργιθεας,16,5,2023,0.000717,0
7,22.02629,39.60300,φαρκαδονας,16,5,2023,0.000464,0
8,22.51098,39.51390,κιλελερ,16,5,2023,0.000369,0
9,22.76105,39.69186,αγιας,16,5,2023,0.000267,0


In [11]:
results_test.to_csv(f"../../data/{NUTS2}/results/{NUTS0}_{NUTS2}_Results_{YEAR}-{MONTH}-{PERIOD}_(new).csv", encoding = enc, index = False)

In [12]:
##TODO Visualisation of results